# Deconstructing motion regressors

**The complaint this notebook is built around.** The standard way to put head motion in a
GLM is to hand it six columns (or twelve, with derivatives) and fit **one β per column**.
That β says: *this voxel's signal responds to head roll with a fixed gain, the same gain
for the whole run, linear in displacement.*

Three separate assumptions are buried in that sentence, and they fail for different reasons:

| assumption | fails because | fixed by MotSim? |
|---|---|---|
| **gain is linear in displacement** | curved intensity edges; nonlinear gradients; motion resamples tissue-class proportions, so even the *sign* of the change is not fixed | **yes** |
| **the column means one thing** | a motion trace mixes slow drift, respiratory *pseudo*-motion, and discrete jerks — three unrelated physical couplings | **no** |
| **the gain is constant over the run** | spin history, B0/shim drift, pose-dependent distortion | **no** |

**What MotSim actually is.** The simulated series at voxel *v* is `f_v(p(t))`, a smooth
nonlinear function of the 6-vector pose. Taylor-expand:

$$f_v(p) \approx f_v(0) + \nabla f_v \cdot p + \tfrac{1}{2}\, p^{\top} H_v\, p + \dots$$

so its temporal PCs span, to second order, the 6 linear terms and the 21 pairwise products
$p_i p_j$. **MotSim is a data-adaptive Friston-24-style expansion in which the anatomy
chooses which products matter.** That is a real answer to row 1 — and it leaves rows 2 and 3
completely untouched, because the PCs are still a deterministic, stationary function of
$p(t)$ and each still gets one β.

Worse, for respiratory pseudo-motion MotSim's forward model is *actively wrong*: chest
expansion perturbs B0, which **warps** the image; it does not move the head. It lands in
$p(t)$ regardless, and MotSim then simulates it as a rigid resample.

**So: is there anything there?** That is an empirical question, and this notebook is built
to answer it on your data rather than to argue about it. It tests four expansions against
the standard model, with two referees and an explicit null.

---

## The plan

| stage | question | how |
|---|---|---|
| 1 | — | moco each run, align runs to each other, one resample, build a mask |
| 2 | — | build the regressor families: 6mot, 12mot, 24mot, MotSim |
| 3 | **Is there a respiratory band to separate?** | power spectrum of the motion parameters |
| 4 | **Is the coupling stationary in time?** | time-varying β via Legendre expansion, nested F, surrogate null |
| 5 | **Is the coupling band-dependent?** | band-split columns, nested F, surrogate null |
| 6 | **Do spikes need their own decay term?** | FD convolved with exponential kernels |
| 7 | **Does any of it generalise?** | leave-one-run-out held-out R² — the referee |
| 8 | **Does it help the thing we care about?** | split-half reliability of the task betas |
| 9 | **Can we afford it?** | fractional ridge on the expansion block instead of selection |
| 10 | **Does MotSim already do it?** | the same expansions applied to the MotSim PCs |

## Two properties that make this honest

**1. Every expansion is exactly nested in the model it is compared against.** That is not
an accident, it is engineered:

- *Time-varying:* the expansion is $x(t)\,P_m(t)$ for Legendre $P_0 \dots P_D$. $P_0$ is the
  constant, so degree 0 returns $x$ itself. Setting $c_1 = c_2 = \dots = 0$ recovers the
  standard model exactly.
- *Band split:* the bands are built by zeroing disjoint, exhaustive FFT bins, so they **sum
  back to the original column**. Setting all band betas equal recovers the standard model
  exactly.

Nesting is what makes the F-test legitimate and what makes "did this help?" a question with
an answer rather than a comparison of two unrelated fits.

**2. In-sample $R^2$ is useless here, and we never use it as evidence.** Patriat measured
this directly: MotSim's explained variance asymptotes to *the slope of random regressors*,
which is the null of $k/T$ per column. Any expansion will raise in-sample $R^2$. So we use:

- a **surrogate null** — phase-randomised motion columns, which preserve the power spectrum
  and hence the autocorrelation exactly, but carry no true relation to the data. This matters
  because residual autocorrelation inflates F and makes textbook p-values meaningless.
- **leave-one-run-out held-out $R^2$**, which needs no null at all: overfitting is punished
  rather than rewarded, so models with different column counts can be compared directly.

## One consequence worth noticing up front

The conventional model fits motion nuisance **per run** (block-diagonal, its own β per run).
That model is **unfalsifiable on held-out data by construction** — a per-run β cannot
transfer to a run it was not fit on. To test whether the motion→signal coupling generalises
at all, we have to force a *shared* β across runs. That is a real change from standard
practice, and it is the only way the question can be asked.

## What you need

- **3 or more raw BOLD runs** (unpreprocessed; this notebook does the motion correction).
  Three is the minimum for leave-one-run-out to mean anything; more is better.
- **One BIDS `*_events.tsv` per run**, in the same order.
- **TR around 1 s.** This is not incidental. At TR = 1 s the Nyquist limit is 0.5 Hz, so the
  respiratory band (~0.15–0.4 Hz) is *resolved* and can be separated. At TR = 2.6 s — the
  Patriat acquisition — respiration aliases down into the BOLD band and there is nothing to
  separate. Stage 3 checks this for your data before Stage 5 tries to use it.

Nothing below runs until `CFG` points at real files. Every cell is written to fail with a
clear message rather than a traceback if it cannot proceed.

In [ ]:
from __future__ import annotations

import warnings
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import torch

from fastfuncstuff.design.bids_events import parse_bids_events
from fastfuncstuff.design.builder import create_onset_matrix_microtime
from fastfuncstuff.design.hrf import get_spmg1_hrf
from fastfuncstuff.design.matrices import convolve_hrf_microtime
from fastfuncstuff.glm.core import construct_polynomial_matrix
from fastfuncstuff.glm.ridge import _fit_ridge_multiple_fracs
from fastfuncstuff.processing.allineate import AffineAlignConfig, allineate
from fastfuncstuff.processing.ffs_moco import MocoConfig, moco, resample_timeseries
from fastfuncstuff.processing.io import load_image
from fastfuncstuff.processing.mask import automask
from fastfuncstuff.processing.motsim import motsim_regressors, parse_motsim_spec
from fastfuncstuff.utils import get_device

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "font.size": 10,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.alpha": 0.25,
        "figure.facecolor": "white",
    }
)
warnings.filterwarnings("ignore", category=UserWarning)
print("torch", torch.__version__)

## Configuration — edit me

`bold` and `events` are the only two you must set. Everything else has a defensible default.

A note on `max_voxels`: the statistics here are cheap per voxel but we run a lot of them
(several nested fits × a surrogate ensemble × a leave-one-run-out loop). Subsampling voxels
inside the mask costs nothing in validity — every test is per-voxel and independent — and
turns a coffee break into a few seconds. Set it to `None` once you know the notebook runs
and you want the full maps.

In [ ]:
CFG = {
    # ── data: the only two you must set ──────────────────────────────────────
    "bold": [
        # "/path/sub-01_task-X_run-1_bold.nii.gz",
        # "/path/sub-01_task-X_run-2_bold.nii.gz",
        # "/path/sub-01_task-X_run-3_bold.nii.gz",
    ],
    "events": [
        # "/path/sub-01_task-X_run-1_events.tsv",
        # "/path/sub-01_task-X_run-2_events.tsv",
        # "/path/sub-01_task-X_run-3_events.tsv",
    ],
    "tr": None,             # seconds; None reads pixdim[4] from the first run
    "device": None,         # None = get_device() (cuda / mps / cpu)

    # ── preprocessing ────────────────────────────────────────────────────────
    "moco_base": "middle",  # "middle" | "first" | int volume index
    "drop_first": 0,        # TRs to discard from the start of every run (T1 settling)
    "mask_dilate": 0,       # outward dilation of the analysis mask, in voxels

    # ── events ───────────────────────────────────────────────────────────────
    "event_ignore": [],     # trial_type values to drop entirely, e.g. ["fixation"]
    "microtime_dt": 0.1,    # sub-TR resolution for the HRF convolution

    # ── model ────────────────────────────────────────────────────────────────
    # polort per run. AFNI's rule is 1 + floor(minutes / 2.5); None applies it.
    "polort": None,
    "motsim": "both,12",    # the MotSim model, as ffs_moco -motsim takes it

    # ── the expansions under test ────────────────────────────────────────────
    "tv_degrees": [0, 1, 2, 3],   # time-varying-beta ladder; 0 == today's model
    # Band edges in Hz. Must be contiguous and exhaustive up to Nyquist, or the
    # bands will not sum back to the original column and nesting breaks. The last
    # edge is replaced by Nyquist automatically.
    "bands": [0.0, 0.01, 0.10, 0.15, 0.50],
    "resp_band": (0.15, 0.50),    # which band gets a Hilbert quadrature partner
    "spin_taus": [2.0, 6.0, 16.0],  # spin-history decay constants, seconds

    # ── statistics ───────────────────────────────────────────────────────────
    "n_surrogates": 20,     # phase-randomised motion ensembles for the null
    "max_voxels": 20000,    # None = every voxel in the mask
    "seed": 0,
}

DEVICE = torch.device(CFG["device"]) if CFG["device"] else get_device()
RNG = np.random.default_rng(CFG["seed"])
torch.manual_seed(CFG["seed"])

HAVE_DATA = len(CFG["bold"]) >= 3 and len(CFG["events"]) == len(CFG["bold"])
if not HAVE_DATA:
    print(
        "CFG['bold'] / CFG['events'] are not set (or fewer than 3 runs).\\n"
        "Every cell below will skip cleanly until they are. Three runs is the\\n"
        "minimum for leave-one-run-out to mean anything."
    )
else:
    print(f"{len(CFG['bold'])} runs, device={DEVICE}")

---

# Stage 1 — Preprocessing

Three steps, and one thing done deliberately:

1. **Motion-correct each run** to its own base volume. We keep the per-volume matrices, not
   just the parameters — MotSim needs the matrices, and so does step 3.
2. **Align the runs to each other**, rigid, on the motion-corrected means. Run 0's grid is
   the common space.
3. **One resample.** Rather than writing a motion-corrected series and then warping it
   again, we compose `M_moco[t] @ M_xrun` and resample the *raw* data once. Two
   interpolations blur the edges, and the edges are precisely where motion artefact lives —
   which would bias every test in this notebook toward "no effect".

We do not slice-time correct, do not smooth, and do not band-pass. Smoothing would mix the
edge voxels that carry the artefact into their neighbours; band-passing would destroy the
frequency structure Stage 5 exists to test.

In [ ]:
def _moco_base_index(n_t: int) -> int:
    spec = CFG["moco_base"]
    if spec == "middle":
        return n_t // 2
    if spec == "first":
        return 0
    return int(spec)


MOCO_CFG = MocoConfig(
    cost="wls",
    twopass=True,        # coarse-blur then fine pass: robust to larger motion
    interp="heptic",
    final_interp="wsinc5",
    compile=False,       # notebook: warmup would dominate
    device=str(DEVICE),
    verb=0,
)

runs = []  # one dict per run, filled below

if HAVE_DATA:
    for i, path in enumerate(CFG["bold"]):
        data, hdr = load_image(path, device=torch.device("cpu"))
        if data.ndim != 4:
            raise ValueError(f"{path} is {data.ndim}D; a 4D run is required")
        if CFG["drop_first"]:
            data = data[CFG["drop_first"] :]
        tr = CFG["tr"] or float(hdr["header"]["pixdim"][4])

        base_idx = _moco_base_index(data.shape[0])
        cfg_i = replace(MOCO_CFG, base_index=base_idx)
        res = moco(data.float(), cfg_i, header_info=hdr)

        runs.append(
            {
                "path": path,
                "raw": data.float(),
                "hdr": hdr,
                "tr": tr,
                "base_idx": base_idx,
                "base_vol": data[base_idx].float().clone(),
                "matrices_vox": res.matrices_vox,   # (T,4,4) base coords -> volume t
                "params": res.params,               # (T,6) DICOM [dx,dy,dz,rz,rx,ry] mm/deg
                "max_disp": res.max_displacement,
                "mean": res.aligned.mean(0),
                "n_t": data.shape[0],
            }
        )
        print(
            f"run {i}: {data.shape[0]} vols, TR={tr:.3f}s, base={base_idx}, "
            f"max displacement {res.max_displacement.max():.2f} mm"
        )

    TR = runs[0]["tr"]
    if len({round(r["tr"], 4) for r in runs}) > 1:
        print(f"WARNING: runs disagree on TR; using run 0's {TR:.3f}s for the design")
    if not 0.4 <= TR <= 1.6:
        print(
            f"NOTE: TR={TR:.2f}s. This notebook's frequency section assumes ~1s. "
            "Stage 3 will tell you whether the respiratory band is resolved."
        )

In [ ]:
XRUN_CFG = AffineAlignConfig(
    dof="rigid",            # same subject, same session: pose only
    cost="lpa",             # same modality on both sides
    source_automask=True,
    autoweight=True,
    final_interp="wsinc5",
    device=str(DEVICE),
    verb=0,
)

if HAVE_DATA:
    ref = runs[0]
    eye = torch.eye(4, dtype=torch.float32)
    for i, r in enumerate(runs):
        if r["raw"].shape[1:] != ref["raw"].shape[1:]:
            raise ValueError(
                f"run {i} has grid {tuple(r['raw'].shape[1:])} but run 0 has "
                f"{tuple(ref['raw'].shape[1:])}. This notebook assumes one acquisition "
                "geometry; resample them to a common grid first (ffs_util_resample)."
            )
        if i == 0:
            r["M_xrun"] = eye.clone()
            continue
        M, _warped = allineate(
            ref["mean"].to(DEVICE),
            r["mean"].to(DEVICE),
            XRUN_CFG,
            base_header=ref["hdr"],
            source_header=r["hdr"],
        )
        r["M_xrun"] = M.detach().cpu().float()
        shift = float(torch.linalg.norm(r["M_xrun"][:3, 3]))
        print(f"run {i} -> run 0: {shift:.2f} voxels of translation in the matrix")

In [ ]:
if HAVE_DATA:
    # ONE resample: compose this run's per-volume moco matrix with its cross-run
    # matrix and pull the RAW data straight onto run 0's grid.
    #
    # Direction check, because getting it backwards is silent and plausible:
    # apply_affine(src, M) samples src at M.x, and moco's own Pass 2 produces
    # aligned[t](x) = raw[t](M_moco[t].x) -- so M_moco maps run-space base coords
    # to volume-t coords, and allineate returns a matrix with the same meaning
    # (base voxels -> source voxels). Composing right-to-left:
    #     x (run-0 space) --M_xrun--> run-k base --M_moco[t]--> run-k volume t
    for i, r in enumerate(runs):
        M_total = r["matrices_vox"] @ r["M_xrun"].unsqueeze(0)
        aligned, _ = resample_timeseries(
            r["raw"], M_total, MOCO_CFG, DEVICE, disable_pbar=True
        )
        r["aligned"] = aligned
        del r["raw"]  # the notebook holds several runs; the raw copy is done with
    print("single-resample complete:", [tuple(r["aligned"].shape) for r in runs])

In [ ]:
if HAVE_DATA:
    grand = torch.stack([r["aligned"].mean(0) for r in runs]).mean(0)

    brain = automask(grand, dilate_extra=CFG["mask_dilate"])
    # Complete data only: a voxel that any run's motion carried outside the FoV is
    # zero in the volumes that lost it, and a column of zeros-then-signal is a step
    # function perfectly correlated with the motion that caused it. Including those
    # voxels would manufacture exactly the effect we are trying to measure.
    coverage = torch.ones_like(brain)
    for r in runs:
        coverage &= r["aligned"].amin(0) > 0
    MASK = brain & coverage

    n_brain, n_mask = int(brain.sum()), int(MASK.sum())
    print(f"automask: {n_brain:,} voxels")
    print(f"  complete-data intersection: {n_mask:,} ({100 * n_mask / max(n_brain, 1):.1f}%)")

    mask_flat = MASK.reshape(-1)
    VOX_IDX = torch.arange(mask_flat.numel())[mask_flat]
    if CFG["max_voxels"] and VOX_IDX.numel() > CFG["max_voxels"]:
        pick = torch.from_numpy(
            RNG.choice(VOX_IDX.numel(), CFG["max_voxels"], replace=False)
        ).sort().values
        VOX_IDX = VOX_IDX[pick]
        print(f"  analysing a random {VOX_IDX.numel():,}-voxel subsample")

    # Y[i] is (T_i, V) for run i -- time-first, because every design below is (T, k).
    Y = [r["aligned"].reshape(r["n_t"], -1)[:, VOX_IDX].contiguous() for r in runs]
    RUN_LENGTHS = [r["n_t"] for r in runs]
    RUN_STARTS = np.cumsum([0] + RUN_LENGTHS[:-1]).tolist()
    T_TOTAL = sum(RUN_LENGTHS)
    print(f"  Y: {len(Y)} runs x (T_i, {Y[0].shape[1]:,} voxels), T_total={T_TOTAL}")

### What to look for

- **Max displacement per run.** Under ~0.5 mm and this notebook will probably find nothing,
  because there is no artefact to model. That is a real result, not a failure — but prefer a
  subject who moved if you have one, and say so when you report what you found.
- **The complete-data fraction.** If the intersection throws away a large share of the
  automask, the runs are poorly aligned to each other or one run has a big translation.
  Check the cross-run translations printed above.
- **The run means, below.** They should be visually indistinguishable. A visible shift means
  the cross-run alignment failed and everything downstream inherits the error.

In [ ]:
def montage(vol, n=6, cmap="gray", vmin=None, vmax=None, title="", mask=None, ax=None):
    """Evenly spaced axial slices of a (nz, ny, nx) volume, in one row."""
    v = vol.detach().cpu().numpy() if torch.is_tensor(vol) else np.asarray(vol)
    zs = np.linspace(v.shape[0] * 0.2, v.shape[0] * 0.8, n).astype(int)
    if ax is None:
        _, ax = plt.subplots(1, n, figsize=(2.0 * n, 2.3))
    ax = np.atleast_1d(ax)
    if vmin is None or vmax is None:
        finite = v[np.isfinite(v)]
        lo, hi = np.percentile(finite, [2, 98]) if finite.size else (0.0, 1.0)
        vmin, vmax = (lo if vmin is None else vmin), (hi if vmax is None else vmax)
    for a, z in zip(ax, zs):
        a.imshow(v[z].T, cmap=cmap, vmin=vmin, vmax=vmax, origin="lower")
        if mask is not None:
            m = mask.detach().cpu().numpy() if torch.is_tensor(mask) else np.asarray(mask)
            a.contour(m[z].T.astype(float), levels=[0.5], colors="tab:cyan", linewidths=0.6)
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
    ax[0].set_ylabel(title, fontsize=9)
    return ax


if HAVE_DATA:
    fig, axes = plt.subplots(len(runs) + 1, 6, figsize=(12, 2.3 * (len(runs) + 1)))
    for i, r in enumerate(runs):
        montage(r["aligned"].mean(0), title=f"run {i} mean", ax=axes[i])
    montage(grand, title="grand mean\\n+ mask", mask=MASK, ax=axes[-1])
    fig.suptitle("Stage 1 QC — the run means should be indistinguishable", y=1.005)
    fig.tight_layout()
    plt.show()

---

# Stage 2 — The regressor families

Four nuisance families, plus framewise displacement for Stage 6.

- **`mot6`** — the six rigid parameters. The minimum.
- **`mot12`** — those plus their temporal derivatives. The de-facto standard.
- **`mot24`** — the Friston expansion: parameters, parameters at *t−1*, and the squares of
  both. This is the existing answer to "one column means more than one thing", and it is
  worth having in the comparison because MotSim is approximately a data-adaptive version of
  it.
- **`motsim`** — temporal PCs of the simulated motion-induced signal changes. Computed from
  each run's *own* base volume and matrices, with the backward pass inheriting that run's
  `MocoConfig`, exactly as `ffs_moco -motsim` does.

Every family is z-scored **per run**. That keeps the betas comparable across columns, keeps
the squares in `mot24` from dominating by scale alone, and makes the ridge in Stage 9
meaningful (a penalty on raw-millimetre and raw-degree columns penalises them unequally for
no reason).

In [ ]:
def zscore(x: torch.Tensor) -> torch.Tensor:
    """Column-wise z-score. Constant columns pass through as zeros."""
    x = x - x.mean(0, keepdim=True)
    s = x.std(0, keepdim=True)
    return x / s.clamp_min(1e-12)


def framewise_displacement(params: np.ndarray, radius: float = 50.0) -> np.ndarray:
    """Power's FD from (T, 6) [dx, dy, dz, rz, rx, ry] in mm and DEGREES.

    Rotations become arc length on a `radius`-mm sphere, which is the convention
    every FD threshold in the literature is quoted against.
    """
    d = np.diff(params, axis=0, prepend=params[:1])
    d = d.copy()
    d[:, 3:] = np.deg2rad(d[:, 3:]) * radius
    return np.abs(d).sum(1)


FAMILIES: dict[str, list[torch.Tensor]] = {}  # name -> per-run (T_i, k) tensors

if HAVE_DATA:
    spec = parse_motsim_spec(CFG["motsim"])
    for r in runs:
        p = torch.from_numpy(r["params"]).float()            # (T, 6)
        d = torch.cat([torch.zeros(1, 6), p[1:] - p[:-1]])   # temporal derivative
        lag = torch.cat([torch.zeros(1, 6), p[:-1]])         # p at t-1

        pz, dz, lz = zscore(p), zscore(d), zscore(lag)
        r["mot6"] = pz
        r["mot12"] = torch.cat([pz, dz], 1)
        r["mot24"] = torch.cat([pz, lz, zscore(pz**2), zscore(lz**2)], 1)

        ms = motsim_regressors(
            r["base_vol"],
            r["matrices_vox"],
            spec,
            DEVICE,
            config=replace(MOCO_CFG, base_index=r["base_idx"]),
            header_info=r["hdr"],
            verb=0,
        )
        r["motsim"] = zscore(ms.pcs.float())
        r["motsim_var"] = ms.var_explained
        r["fd"] = framewise_displacement(r["params"])

    for name in ("mot6", "mot12", "mot24", "motsim"):
        FAMILIES[name] = [r[name] for r in runs]
        print(f"{name:8s} {FAMILIES[name][0].shape[1]:3d} columns per run")
    print(
        f"\\nMotSim ({spec}) cumulative variance of the simulated series: "
        f"{100 * float(runs[0]['motsim_var'].sum()):.1f}%"
    )

In [ ]:
if HAVE_DATA:
    labels = ["dx", "dy", "dz", "rz", "rx", "ry"]
    fig, axes = plt.subplots(2, len(runs), figsize=(4.2 * len(runs), 5), squeeze=False)
    for i, r in enumerate(runs):
        t = np.arange(r["n_t"]) * TR
        for j, lab in enumerate(labels):
            axes[0, i].plot(t, r["params"][:, j], lw=0.8, label=lab)
        axes[0, i].set_title(f"run {i} — motion parameters")
        axes[0, i].set_ylabel("mm / deg" if i == 0 else "")
        if i == 0:
            axes[0, i].legend(ncol=3, fontsize=7, loc="upper left")

        axes[1, i].plot(t, r["fd"], lw=0.8, color="tab:red")
        axes[1, i].axhline(0.2, ls="--", lw=0.8, color="k")
        axes[1, i].axhline(0.5, ls=":", lw=0.8, color="k")
        axes[1, i].set_xlabel("time (s)")
        axes[1, i].set_ylabel("FD (mm)" if i == 0 else "")
        axes[1, i].set_title(
            f"FD — median {np.median(r['fd']):.3f}, "
            f"{100 * np.mean(r['fd'] > 0.2):.0f}% over 0.2mm"
        )
    fig.tight_layout()
    plt.show()

### What to look for

- **Steps vs. drifts vs. oscillation.** A slow monotone drift in `dz` is the head settling
  or the table relaxing — linear in displacement, well handled by the standard model. A
  *step* is a real reposition and is where spin history lives (Stage 6). A visible fast
  oscillation, especially in the phase-encode-direction translation, is very likely
  respiration rather than motion (Stage 3 confirms it).
- **FD relative to 0.2 mm (dashed) and 0.5 mm (dotted).** These are the conventional
  censoring thresholds. If almost nothing crosses them, the spike section will find nothing.
- **Whether the runs look alike.** If run 3 is visibly worse than runs 1–2, the
  leave-one-run-out numbers in Stage 7 will be dominated by which run was held out. The
  per-fold breakdown there will show you if that happened.

---

# Stage 3 — Is there a respiratory band to separate?

**This gate decides whether Stage 5 can work at all**, so it comes first.

Respiration modulates B0, which shifts and warps the EPI; realignment reports that as
apparent head motion, mostly along the phase-encode direction. Real respiration sits near
0.2–0.35 Hz in adults. Whether you can *see* it depends entirely on TR:

- **TR = 1 s → Nyquist 0.5 Hz.** Respiration is resolved. A distinct peak should be visible,
  and a band split can separate it from everything else.
- **TR = 2.6 s → Nyquist 0.19 Hz.** Respiration aliases to some arbitrary low frequency,
  lands inside the BOLD band, and is *unrecoverable* — no filter separates it, because the
  information is gone.

The plot below marks Nyquist and shades the band edges from `CFG["bands"]`.

In [ ]:
def welch_psd(x: np.ndarray, tr: float, nperseg: int | None = None):
    """Welch PSD of each column of (T, k). Hann window, 50% overlap, no scipy needed."""
    T = x.shape[0]
    nperseg = nperseg or min(256, T)
    step = nperseg // 2
    win = np.hanning(nperseg)[:, None]
    segs = [x[s : s + nperseg] for s in range(0, T - nperseg + 1, step)]
    if not segs:
        segs = [np.pad(x, ((0, nperseg - T), (0, 0)))]
    P = np.zeros((nperseg // 2 + 1, x.shape[1]))
    for s in segs:
        s = (s - s.mean(0, keepdims=True)) * win
        P += np.abs(np.fft.rfft(s, axis=0)) ** 2
    P /= len(segs) * (win**2).sum()
    return np.fft.rfftfreq(nperseg, d=tr), P


if HAVE_DATA:
    nyq = 0.5 / TR
    fig, axes = plt.subplots(1, len(runs), figsize=(4.4 * len(runs), 3.4), squeeze=False)
    for i, r in enumerate(runs):
        f, P = welch_psd(r["params"], TR)
        for j, lab in enumerate(labels):
            axes[0, i].semilogy(f, P[:, j], lw=0.9, label=lab)
        for lo, hi in zip(CFG["bands"][:-1], CFG["bands"][1:]):
            axes[0, i].axvspan(lo, min(hi, nyq), alpha=0.06, color="tab:blue")
        for e in CFG["bands"]:
            if e < nyq:
                axes[0, i].axvline(e, lw=0.6, ls=":", color="0.4")
        axes[0, i].axvspan(*CFG["resp_band"], alpha=0.12, color="tab:orange")
        axes[0, i].set_xlim(0, nyq)
        axes[0, i].set_xlabel("Hz")
        axes[0, i].set_title(f"run {i} — Nyquist {nyq:.2f} Hz")
        if i == 0:
            axes[0, i].set_ylabel("power")
            axes[0, i].legend(ncol=3, fontsize=7)
    fig.suptitle(
        "Motion-parameter spectra. Orange = the putative respiratory band.", y=1.02
    )
    fig.tight_layout()
    plt.show()

    if nyq < 0.25:
        print(
            f"GATE FAILED: Nyquist is {nyq:.2f} Hz (TR={TR:.2f}s). Respiration is aliased\\n"
            "into the low-frequency range and cannot be separated by any filter. Stage 5\\n"
            "will still run -- the bands are still exactly nested, so the F-test is valid --\\n"
            "but do NOT interpret a band effect here as respiratory."
        )
    else:
        band_pow = P[(f >= CFG["resp_band"][0]) & (f < CFG["resp_band"][1])].sum()
        print(
            f"Nyquist {nyq:.2f} Hz: the respiratory band is resolved.\\n"
            f"Run {len(runs) - 1} puts {100 * band_pow / P.sum():.1f}% of its motion-parameter "
            "power in that band."
        )

### What to look for

**A narrow peak between 0.15 and 0.4 Hz, strongest in one translation axis.** That is
respiratory pseudo-motion, and it is the single best argument for the frequency split: it is
not head motion at all, and a model that gives it the same β as a genuine reposition is
averaging two unrelated physical couplings.

**A broad 1/f ramp with no peak.** Slow drift and nothing else. The band split will still be
a valid nested test but has less to find, and any effect it does show is not respiration.

**Power piling up right at Nyquist.** Something faster than you can sample is folding back.
Whatever Stage 5 reports about the top band, it is aliased.

---

# Interlude — the design, and the statistical machinery

## The design

Every model in this notebook has the same skeleton:

```
X = [ task | per-run polynomials | motion-family columns ]
```

Task and polynomials are **identical in every comparison** — they are there so the nuisance
comparison is not contaminated by task variance or drift, not because we are testing them.
Polynomials are per-run Legendre ([[Legendre polynomials]]), block-diagonal, degree from
AFNI's rule `1 + floor(minutes / 2.5)`.

## Three ways to score a model, and what each is worth

1. **In-sample $R^2$ — worth nothing on its own.** Reported only so you can watch it rise
   monotonically as columns are added, which is the point.
2. **Nested F with a surrogate null.** The F-statistic is fine; its *textbook p-value* is
   not, because unfiltered fMRI residuals are heavily autocorrelated and that inflates F by
   an unknown factor. So we compute the same F on **phase-randomised** motion columns —
   surrogates that preserve the power spectrum exactly, therefore the autocorrelation
   exactly, but carry no true relation to the data. The observed F is interpretable against
   *that* distribution.
3. **Leave-one-run-out held-out $R^2$ — the referee.** No null needed, no DoF correction
   needed: a model that buys its fit with extra columns is punished, not rewarded.

## One decision that makes LORO possible

Standard practice fits motion nuisance per run with its own β. That model cannot be tested
on held-out data — a per-run β has nothing to transfer. So for the LORO sections we force a
**shared β across runs**, and residualise each run against its *own* polynomials and task
first (fold-local by construction, which is the usual [[LORO cross-validation]] trap).

That the conventional model is unfalsifiable this way is itself a finding worth sitting
with.

In [ ]:
def _orth_basis(X: torch.Tensor, rtol: float = 1e-8):
    """Orthonormal basis for col(X) plus its numerical rank.

    SVD rather than QR on purpose: these designs are rank-deficient often enough
    (block-diagonal nuisance, a constant column, a collinear derivative) and a
    reduced QR's Q can span MORE than col(X) when R has a zero on the diagonal,
    which silently inflates every projection built from it.
    """
    U, S, _ = torch.linalg.svd(X.double(), full_matrices=False)
    keep = S > rtol * S[0].clamp_min(1e-300)
    return U[:, keep], int(keep.sum())


def rss_and_rank(Y: torch.Tensor, X: torch.Tensor):
    """Residual sum of squares per voxel, and the design's rank.

    Y is (T, V), X is (T, k). The basis is found in float64 (small, and the place
    where rank decisions are made) and applied in Y's dtype (large).
    """
    U, rank = _orth_basis(X)
    U = U.to(Y.dtype).to(Y.device)
    fit = U.T @ Y
    return (Y * Y).sum(0) - (fit * fit).sum(0), rank


def nested_F(Y: torch.Tensor, X0: torch.Tensor, X1: torch.Tensor):
    """F for X1 over the nested X0. Returns (F per voxel, df1, df2)."""
    rss0, r0 = rss_and_rank(Y, X0)
    rss1, r1 = rss_and_rank(Y, X1)
    # Nesting is the whole basis of the test, so check it rather than assume it.
    slack = (rss1 - rss0).max() / rss0.median().clamp_min(1e-30)
    if slack > 1e-6:
        raise ValueError(
            f"X0 is not nested in X1 (residuals grew by {float(slack):.2e} relative). "
            "Every expansion in this notebook must contain the model it is compared to."
        )
    df1, df2 = max(r1 - r0, 1), Y.shape[0] - r1
    return ((rss0 - rss1) / df1) / (rss1 / df2).clamp_min(1e-30), df1, df2


def r2_in_sample(Y: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
    rss, _ = rss_and_rank(Y, X)
    return 1.0 - rss / (Y * Y).sum(0).clamp_min(1e-30)


def residualize(Y: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """Project B out of Y. B is this run's own baseline, so this is fold-local."""
    U, _ = _orth_basis(B)
    U = U.to(Y.dtype).to(Y.device)
    return Y - U @ (U.T @ Y)


def phase_randomize(x: np.ndarray, rng) -> np.ndarray:
    """Surrogate with the SAME power spectrum (hence autocorrelation) and no
    true relation to the data. Each column gets its own random phase."""
    T = x.shape[0]
    X = np.fft.rfft(x - x.mean(0, keepdims=True), axis=0)
    phase = rng.uniform(0, 2 * np.pi, size=X.shape)
    phase[0] = 0.0                      # DC stays real
    if T % 2 == 0:
        phase[-1] = 0.0                 # so does Nyquist
    return np.fft.irfft(np.abs(X) * np.exp(1j * phase), n=T, axis=0)


def surrogate_verdict(observed: float, surrogate: np.ndarray) -> str:
    """Place an observed statistic in the surrogate ensemble, non-parametrically.

    Deliberately NOT a z-score. The surrogate F distribution is right-skewed
    whenever the artefact is low-rank -- one lucky phase draw correlates with the
    shared temporal component and moves every voxel at once -- so a mean and a
    standard deviation describe it badly. "Exceeds 20 of 20" is the honest claim.
    """
    n = len(surrogate)
    return (
        f"{observed:7.3f} vs surrogate median {np.median(surrogate):6.3f} "
        f"[{surrogate.min():.3f}, {surrogate.max():.3f}] -- exceeds {int((surrogate < observed).sum())}/{n}"
    )


print("statistical helpers defined")

### This machinery was checked against ground truth

Before pointing any of it at real data it was run on synthetic series with a **known**
answer: three runs sharing one spatial β map, realistic motion columns (1/f drift + a 0.30 Hz
oscillation + a step), and one of three couplings imposed by hand.

| truth in the data | time-varying test | band-split test | LORO: `+tv1` | LORO: `+bands` |
|---|---|---|---|---|
| stationary, broadband | quiet ✓ | quiet ✓ | **−0.009** ✓ | **−0.079** ✓ |
| gain drifts linearly | **fires** ✓ | quiet ✓ | **+0.148** ✓ | −0.209 ✓ |
| coupling in one band | quiet ✓ | *misses* ✗ | −0.062 ✓ | **+0.481** ✓ |

**The leave-one-run-out referee got all six right.** Positive gain exactly where the effect
was planted, negative everywhere else — including the cross-terms, where the wrong expansion
is correctly punished rather than merely unhelpful.

**The surrogate F screen got five of six**, and the one it missed is worth understanding,
because you may hit it:

> When the artefact is **low-rank and narrowband** — one temporal component shared across
> voxels, concentrated in a narrow frequency band — a phase-randomised surrogate occasionally
> lands a chance correlation with that component, and because the component is shared, that
> single lucky draw raises F at *every voxel at once*. The surrogate null inflates, and a
> real effect can sit inside it.

Respiratory pseudo-motion is precisely low-rank and narrowband. So:

- The surrogate screen is **conservative**: it does not manufacture effects (it was quiet in
  every null condition), but it can hide one.
- **A quiet Stage 5 does not license skipping Stage 7.** Run the leave-one-run-out check
  regardless of what the F screen says.
- This is also why the surrogate result is reported as *"exceeds N of M"* rather than as a
  z-score. That distribution is right-skewed in exactly this situation, and a mean ± SD
  describes it badly.

In [ ]:
def block_diag_runs(per_run: list[torch.Tensor]) -> torch.Tensor:
    """Stack per-run (T_i, k_i) blocks into (T_total, sum k_i), zero-padded off
    the diagonal -- the conventional per-run nuisance layout."""
    out = torch.zeros(T_TOTAL, sum(b.shape[1] for b in per_run))
    t = c = 0
    for b in per_run:
        out[t : t + b.shape[0], c : c + b.shape[1]] = b
        t += b.shape[0]
        c += b.shape[1]
    return out


def stack_runs(per_run: list[torch.Tensor]) -> torch.Tensor:
    """Concatenate per-run (T_i, k) blocks in time -- the SHARED-beta layout."""
    return torch.cat(per_run, 0)


if HAVE_DATA:
    polort = CFG["polort"]
    if polort is None:
        polort = int(1 + np.floor((RUN_LENGTHS[0] * TR / 60.0) / 2.5))
    POLY = [
        construct_polynomial_matrix(n, polort, device=torch.device("cpu"))
        for n in RUN_LENGTHS
    ]
    print(f"polort {polort} ({polort + 1} columns per run)")

    all_onsets, durations, cond_labels = parse_bids_events(
        CFG["events"], event_ignore=CFG["event_ignore"] or None, n_runs=len(runs)
    )
    dt = CFG["microtime_dt"]
    onset_micro = create_onset_matrix_microtime(
        all_onsets, RUN_STARTS, TR, T_TOTAL, dt, durations, torch.device("cpu")
    )
    hrf = get_spmg1_hrf(microtime_dt=dt, device=torch.device("cpu"))
    TASK = convolve_hrf_microtime(
        onset_micro, hrf, T_TOTAL, tr=TR, microtime_dt=dt,
        run_starts=RUN_STARTS, device=torch.device("cpu"),
    ).float()
    TASK_PER_RUN = [TASK[s : s + n] for s, n in zip(RUN_STARTS, RUN_LENGTHS)]
    print(f"task design: {TASK.shape[1]} conditions {cond_labels}, durations {durations}")

    # The baseline every model shares, in both layouts.
    BASE_BLOCK = torch.cat([TASK, block_diag_runs(POLY)], 1)
    BASE_PER_RUN = [torch.cat([t, p], 1) for t, p in zip(TASK_PER_RUN, POLY)]

    YCAT = torch.cat(Y, 0)   # (T_total, V) for the in-sample tests
    print(f"baseline design: {BASE_BLOCK.shape[1]} columns, YCAT {tuple(YCAT.shape)}")

In [ ]:
if HAVE_DATA:
    # The tests below run a few hundred projections over every voxel, so the data
    # goes to the device once rather than per fit.
    YCAT = YCAT.to(DEVICE)
    Y_DEV = [y.to(DEVICE) for y in Y]
    print(f"YCAT on {YCAT.device}: {YCAT.element_size() * YCAT.nelement() / 1e6:.0f} MB")

---

# Stage 4 — Is the coupling stationary in time?

## The model

Instead of a single β per column, let the coefficient itself vary over the run:

$$y(t) \;=\; \beta(t)\,x(t) + \dots, \qquad \beta(t) = \sum_{m=0}^{D} c_m P_m(t)$$

which is just a design with columns $x(t)P_m(t)$ for Legendre $P_0 \dots P_D$. Two reasons
for Legendre rather than the block-splitting you might reach for first:

- **No arbitrary boundaries.** A hard block edge introduces a discontinuity in the modelled
  gain, which is itself a step function — exactly the kind of thing we are trying to detect.
- **Exact nesting, and a ladder.** $P_0$ is the constant, so degree 0 *is* today's model.
  Degree 1 adds "the gain drifts linearly", degree 2 adds curvature. Each rung contains the
  one below it.

Cost is $(D+1)\times$ the columns. We run the whole ladder so you can see where it stops
paying.

## Why the gain might drift

B0 and shim drift over a run; as the head slowly leaves its starting pose, the local
linearisation of a genuinely nonlinear pose→signal map changes; and spin-history effects
depend on recent history rather than current position. All three are real; whether any is
*large* is what we are measuring.

In [ ]:
def expand_time_varying(C: torch.Tensor, degree: int) -> torch.Tensor:
    """(T, k) -> (T, k*(degree+1)): each column times each Legendre P_m(t).

    P_0 is the constant, so degree=0 returns C unchanged -- which is what makes
    the ladder exactly nested and degree 0 literally today's model.
    """
    P = construct_polynomial_matrix(C.shape[0], degree, device=C.device, dtype=C.dtype)
    return torch.cat([C * P[:, m : m + 1] for m in range(P.shape[1])], 1)


def tv_design(per_run_cols: list[torch.Tensor], degree: int) -> torch.Tensor:
    """Baseline + per-run block-diagonal motion columns, time-expanded."""
    return torch.cat(
        [BASE_BLOCK, block_diag_runs([expand_time_varying(c, degree) for c in per_run_cols])],
        1,
    ).to(DEVICE)


def surrogate_runs(per_run_cols: list[torch.Tensor]) -> list[torch.Tensor]:
    """Phase-randomised twins: same spectrum, same autocorrelation, no truth."""
    return [
        torch.from_numpy(phase_randomize(c.cpu().numpy(), RNG)).float() for c in per_run_cols
    ]


TV_FAMILY = "mot12"   # the model the field actually uses; change to compare others
tv = {}

if HAVE_DATA:
    cols = FAMILIES[TV_FAMILY]
    degrees = CFG["tv_degrees"]
    designs = {d: tv_design(cols, d) for d in degrees}

    print(f"{'degree':>6} {'columns':>8} {'in-sample R2':>13} {'F vs degree 0':>28}")
    for d in degrees:
        r2 = float(r2_in_sample(YCAT, designs[d]).mean())
        if d == degrees[0]:
            line = "        (reference)"
        else:
            F, df1, df2 = nested_F(YCAT, designs[degrees[0]], designs[d])
            tv[d] = F.cpu()
            line = (
                f"median {float(F.median()):6.3f}  "
                f"p95 {float(F.quantile(0.95)):6.2f}  df=({df1},{df2})"
            )
        print(f"{d:>6} {designs[d].shape[1]:>8} {r2:>13.4f}  {line}")

In [ ]:
SUR_TV = {}

if HAVE_DATA:
    # The null: the same ladder, on columns that have this motion's spectrum but
    # no relationship to the data. Anything the real columns do beyond THIS is the
    # part that is about the brain rather than about autocorrelation and extra
    # degrees of freedom.
    for d in CFG["tv_degrees"][1:]:
        med, p95 = [], []
        for _ in range(CFG["n_surrogates"]):
            s = surrogate_runs(FAMILIES[TV_FAMILY])
            F, _, _ = nested_F(YCAT, tv_design(s, CFG["tv_degrees"][0]), tv_design(s, d))
            med.append(float(F.median()))
            p95.append(float(F.quantile(0.95)))
        SUR_TV[d] = (np.array(med), np.array(p95))
        print(f"degree {d}: median F {surrogate_verdict(float(tv[d].median()), SUR_TV[d][0])}")
        print(f"{'':10s} p95 F    {surrogate_verdict(float(tv[d].quantile(0.95)), SUR_TV[d][1])}")

In [ ]:
if HAVE_DATA and tv:
    d_show = sorted(tv)[0]
    F = tv[d_show]
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))

    ax[0].hist(F.numpy(), bins=80, density=True, alpha=0.7, label="observed")
    for m in SUR_TV[d_show][0]:
        ax[0].axvline(m, color="tab:red", lw=0.5, alpha=0.5)
    ax[0].axvline(float(F.median()), color="k", lw=1.6, label="observed median")
    ax[0].axvline(SUR_TV[d_show][0][0], color="tab:red", lw=0.5, label="surrogate medians")
    ax[0].set_xlim(0, float(F.quantile(0.995)))
    ax[0].set_xlabel(f"F (degree {d_show} vs 0)"); ax[0].set_ylabel("density")
    ax[0].set_title("observed F vs the surrogate null"); ax[0].legend(fontsize=8)

    for d in sorted(tv):
        ax[1].plot(
            np.sort(tv[d].numpy()), np.linspace(0, 1, tv[d].numel()), label=f"degree {d}"
        )
        ax[1].axvline(SUR_TV[d][0].mean(), ls="--", lw=0.8, color="0.5")
    ax[1].set_xlim(0, float(max(t.quantile(0.99) for t in tv.values())))
    ax[1].set_xlabel("F"); ax[1].set_ylabel("cumulative fraction of voxels")
    ax[1].set_title("the ladder (dashed = surrogate medians)"); ax[1].legend(fontsize=8)

    # Where is it? A gain that drifts should not be uniformly distributed -- if it
    # is real it should sit where motion artefact sits: brain edges and frontal.
    fmap = torch.zeros(MASK.numel()); fmap[VOX_IDX] = F
    ax[2].axis("off")
    fig.tight_layout(); plt.show()

    montage(
        fmap.reshape(MASK.shape), n=6, cmap="inferno",
        vmin=0, vmax=float(F.quantile(0.98)), mask=MASK,
        title=f"F, degree {d_show} vs 0",
    )
    plt.suptitle("Where the coupling looks non-stationary", y=1.05)
    plt.show()

### What to look for

**Compare the observed median F to the red surrogate lines, not to 1.0.** An F of 2 sounds
convincing until the surrogates also sit at 2, which is what autocorrelated residuals plus
extra columns will do to you.

**Look at the map, not just the histogram.** This is the part a summary statistic cannot
tell you. If the non-stationarity is real it should be *anatomically patterned* — brain
edges, frontal cortex, near the sinuses, wherever your motion artefact already lives. A map
that looks like salt-and-pepper noise at the same level everywhere is the extra-columns
effect, however big the F.

**Watch where the ladder stops paying.** If degree 1 clears the surrogate and degree 2 does
not, the gain drifts roughly linearly and one extra column per regressor buys it. If nothing
clears, the stationarity assumption is not costing you anything on this data — which is a
genuine, publishable, load-off-your-mind answer.

**A caveat that matters.** A drifting β is not proof that the *coupling* drifts. It is also
what you would see if the *motion estimate* got better or worse over the run — moco is
anchored to one base volume, and a run that drifts far from it is being registered across a
larger displacement by the end. You cannot separate these two here. Stage 10 gives a partial
handle: MotSim is built from the same estimates, so if the drift is an estimation artefact it
should follow MotSim too.

---

# Stage 5 — Is the coupling band-dependent?

Split every motion column into contiguous frequency bands and give each its own β. Because
the bands are disjoint and exhaustive FFT bins, **they sum back to the original column
exactly**, so setting all band betas equal recovers today's model — nested, again.

This is the test with the clearest mechanistic story, because the bands are *different
physics*:

| band | what is in it | how it couples to the signal |
|---|---|---|
| < 0.01 Hz | head settling, table relaxation, gradient heating | geometry; the standard linear model is reasonable here |
| 0.01–0.10 Hz | real slow motion, and the BOLD band | geometry, but confounded with actual signal |
| 0.15–0.50 Hz | respiration — largely **not motion at all** | B0 modulation warps the image; nothing is displaced |

One β across all of that is an average of unrelated couplings. The respiratory band also
gets a **Hilbert quadrature** partner, because its phase relative to acquisition is
arbitrary and a sine alone cannot fit an arbitrary phase (the same reason RETROICOR uses
sine/cosine pairs).

In [ ]:
def band_split(x: torch.Tensor, tr: float, edges: list[float]) -> list[torch.Tensor]:
    """(T, k) -> one (T, k) per band. The bands sum back to x EXACTLY, which is
    what makes the band model contain the unsplit one."""
    T = x.shape[0]
    xn = x.cpu().numpy()
    f = np.fft.rfftfreq(T, d=tr)
    X = np.fft.rfft(xn, axis=0)
    ed = list(edges)
    ed[-1] = max(ed[-1], f[-1] + 1e-9)   # the top edge must reach Nyquist or power leaks out
    out = []
    for lo, hi in zip(ed[:-1], ed[1:]):
        m = (f >= lo) & (f < hi)
        Xb = np.zeros_like(X)
        Xb[m] = X[m]
        out.append(torch.from_numpy(np.fft.irfft(Xb, n=T, axis=0)).float())
    return out


def hilbert_quadrature(x: torch.Tensor) -> torch.Tensor:
    """Imaginary part of the analytic signal: the 90-degree-shifted twin, so a
    band's arbitrary phase relative to acquisition becomes fittable."""
    T = x.shape[0]
    X = np.fft.fft(x.cpu().numpy(), axis=0)
    h = np.zeros(T)
    h[0] = 1.0
    if T % 2 == 0:
        h[T // 2] = 1.0
        h[1 : T // 2] = 2.0
    else:
        h[1 : (T + 1) // 2] = 2.0
    return torch.from_numpy(np.imag(np.fft.ifft(X * h[:, None], axis=0))).float()


def band_cols(C: torch.Tensor, with_quadrature: bool = True) -> torch.Tensor:
    bands = band_split(C, TR, CFG["bands"])
    recon = torch.stack(bands).sum(0)
    assert torch.allclose(recon, C, atol=1e-4), "bands must sum to the original column"
    if with_quadrature:
        lo, hi = CFG["resp_band"]
        edges = CFG["bands"]
        for j, (a, b) in enumerate(zip(edges[:-1], edges[1:])):
            if a >= lo - 1e-9 and b <= hi + 1e-9:
                bands.append(hilbert_quadrature(bands[j]))
    return torch.cat([zscore(b) for b in bands], 1)


def band_design(per_run_cols, split: bool, quad: bool = True) -> torch.Tensor:
    blocks = [band_cols(c, quad) if split else c for c in per_run_cols]
    return torch.cat([BASE_BLOCK, block_diag_runs(blocks)], 1).to(DEVICE)


band = {}
if HAVE_DATA:
    cols = FAMILIES[TV_FAMILY]
    X0 = band_design(cols, split=False)
    for quad in (False, True):
        X1 = band_design(cols, split=True, quad=quad)
        F, df1, df2 = nested_F(YCAT, X0, X1)
        band[quad] = F.cpu()
        sur_med = []
        for _ in range(CFG["n_surrogates"]):
            s = surrogate_runs(cols)
            Fs, _, _ = nested_F(
                YCAT, band_design(s, split=False), band_design(s, split=True, quad=quad)
            )
            sur_med.append(float(Fs.median()))
        sur_med = np.array(sur_med)
        tag = "bands + resp quadrature" if quad else "bands only"
        print(
            f"{tag:24s} {X1.shape[1]:4d} cols  df=({df1},{df2})\n"
            f"{'':24s} median F {surrogate_verdict(float(F.median()), sur_med)}"
        )

In [ ]:
if HAVE_DATA and band:
    F = band[True]
    fmap = torch.zeros(MASK.numel()); fmap[VOX_IDX] = F
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
    ax[0].hist(band[False].numpy(), bins=80, alpha=0.6, density=True, label="bands only")
    ax[0].hist(F.numpy(), bins=80, alpha=0.6, density=True, label="+ resp quadrature")
    ax[0].set_xlim(0, float(F.quantile(0.995)))
    ax[0].set_xlabel("F vs unsplit"); ax[0].legend(fontsize=8)
    ax[0].set_title("band split")

    # Which band is doing the work? Fit the split model once and read the betas.
    Xb = band_design(FAMILIES[TV_FAMILY], split=True, quad=True)
    beta = torch.linalg.pinv(Xb.double()).to(YCAT.dtype) @ YCAT
    n_nuis = Xb.shape[1] - BASE_BLOCK.shape[1]
    per_run_k = n_nuis // len(runs)
    n_bands = len(CFG["bands"]) - 1
    k_family = FAMILIES[TV_FAMILY][0].shape[1]
    b = beta[BASE_BLOCK.shape[1] :].abs().mean(1).reshape(len(runs), -1)[0]
    grouped = b[: n_bands * k_family].reshape(n_bands, k_family).mean(1)
    ax[1].bar(range(n_bands), grouped.cpu().numpy())
    ax[1].set_xticks(range(n_bands))
    ax[1].set_xticklabels(
        [f"{a:g}-{c:g}" for a, c in zip(CFG["bands"][:-1], CFG["bands"][1:])],
        rotation=30, fontsize=8,
    )
    ax[1].set_ylabel("mean |beta|"); ax[1].set_title("run 0: which band carries the weight")
    fig.tight_layout(); plt.show()

    montage(
        fmap.reshape(MASK.shape), n=6, cmap="inferno",
        vmin=0, vmax=float(F.quantile(0.98)), mask=MASK, title="F, bands vs unsplit",
    )
    plt.suptitle("Where the coupling looks band-dependent", y=1.05)
    plt.show()

### What to look for

**Does the quadrature term help?** If "+ resp quadrature" moves the F distribution
noticeably above "bands only", you have phase-locked structure in the respiratory band —
strong evidence that band is carrying something physiological rather than head motion.

**Which band carries the weight.** If the low band dominates, the split is buying you
little that a polynomial did not already have. If the respiratory band has weight comparable
to the low band, the averaging argument is real on your data.

**Cross-check against Stage 3.** A band effect at frequencies where the spectrum showed no
distinguishable structure is suspicious. And if the Stage 3 gate failed on TR, an effect in
the top band is aliased and must not be called respiration.

**Do not stop here if this is quiet.** This is the test the ground-truth check caught
under-powered: a low-rank narrowband artefact — which respiratory pseudo-motion is, exactly —
inflates its own surrogate null, because one lucky phase draw moves every voxel together. In
the synthetic case where the coupling was planted entirely in one band, this screen said
nothing and the Stage 7 referee found **+0.48** held-out $R^2$. Run Stage 7 regardless.

---

# Stage 6 — Spikes and spin history

A jerk does not produce an instantaneous signal change. It changes the magnetisation history
of the slices involved, and that perturbation **decays over several TRs**. A column that is
nonzero only at the moment of the jerk cannot express that, whatever β it gets.

So: take framewise displacement, convolve it with exponential decays at a few time constants,
and ask whether those columns explain variance beyond the motion parameters. This is the one
mechanism the MotSim paper explicitly says its model cannot reach.

In [ ]:
def spin_history_regressors(fd: np.ndarray, tr: float, taus: list[float]) -> torch.Tensor:
    """FD convolved with exp(-t/tau). Each column is 'how much recent disturbance
    is this volume still carrying', at one memory length."""
    T = len(fd)
    out = []
    for tau in taus:
        n = int(np.ceil(5 * tau / tr)) + 1
        k = np.exp(-np.arange(n) * tr / tau)
        out.append(np.convolve(fd, k)[:T])
    return zscore(torch.from_numpy(np.stack(out, 1)).float())


spin = {}
if HAVE_DATA:
    for r in runs:
        r["spin"] = spin_history_regressors(r["fd"], TR, CFG["spin_taus"])
    FAMILIES["spin"] = [r["spin"] for r in runs]

    X0 = torch.cat([BASE_BLOCK, block_diag_runs(FAMILIES[TV_FAMILY])], 1).to(DEVICE)
    X1 = torch.cat(
        [
            BASE_BLOCK,
            block_diag_runs([torch.cat([a, b], 1) for a, b in zip(FAMILIES[TV_FAMILY], FAMILIES["spin"])]),
        ],
        1,
    ).to(DEVICE)
    F, df1, df2 = nested_F(YCAT, X0, X1)
    spin["F"] = F.cpu()

    sur_med = []
    for _ in range(CFG["n_surrogates"]):
        s = surrogate_runs(FAMILIES["spin"])
        Xs = torch.cat(
            [BASE_BLOCK, block_diag_runs([torch.cat([a, b], 1) for a, b in zip(FAMILIES[TV_FAMILY], s)])], 1
        ).to(DEVICE)
        Fs, _, _ = nested_F(YCAT, X0, Xs)
        sur_med.append(float(Fs.median()))
    sur_med = np.array(sur_med)
    print(
        f"spin history (tau={CFG['spin_taus']}s, df=({df1},{df2}))\n"
        f"  median F {surrogate_verdict(float(F.median()), sur_med)}"
    )
    frac_big = float((torch.from_numpy(np.concatenate([r["fd"] for r in runs])) > 0.2).float().mean())
    print(f"{100 * frac_big:.1f}% of volumes exceed FD 0.2mm -- if this is ~0, expect nothing here")

### What to look for

**Read this one against the FD trace from Stage 2.** If almost no volumes cross 0.2 mm,
there are no spikes and a null result here says nothing about spin history — only that this
subject did not jerk.

**If it does clear the surrogates, note which τ.** A short τ (~2 s) is consistent with
through-plane spin history. A long τ (~16 s) is more likely to be a slow post-movement
settling of the field, which is a different problem with a different fix.

---

# Stage 7 — The referee: leave-one-run-out

Everything above is in-sample. This is not.

For each held-out run: residualise every run against **its own** task and polynomials (the
classic [[LORO cross-validation]] trap is projecting a baseline estimated from the full
dataset, which leaks the held-out run into the fit — doing it per run makes it fold-local by
construction). Fit a **shared** β on the remaining runs. Predict the held-out run from *its*
motion, which the model has never seen. Score $R^2$.

Two things this buys that nothing above does:

- **No degrees-of-freedom correction is needed.** A model that bought its fit with extra
  columns predicts *worse*, not better. `mot24` and `mot12+tv2` can be compared directly
  even though one has three times the columns.
- **Negative $R^2$ is meaningful.** It means the coupling learned on other runs actively
  hurts on this one — the model transfers worse than predicting nothing at all.

In [ ]:
def loro_r2(Yr: list[torch.Tensor], Cr: list[torch.Tensor]):
    """Shared-beta leave-one-run-out. Both lists are already residualised per run.
    Returns (mean R2 per voxel, per-fold mean R2)."""
    n = len(Yr)
    folds = []
    for k in range(n):
        Xtr = torch.cat([Cr[i] for i in range(n) if i != k], 0)
        Ytr = torch.cat([Yr[i] for i in range(n) if i != k], 0)
        beta = torch.linalg.pinv(Xtr.double()).to(Ytr.dtype) @ Ytr
        resid = Yr[k] - Cr[k] @ beta
        folds.append(1.0 - (resid**2).sum(0) / (Yr[k] ** 2).sum(0).clamp_min(1e-30))
    stacked = torch.stack(folds)
    return stacked.mean(0), stacked.mean(1)


if HAVE_DATA:
    BASE_DEV = [b.to(DEVICE) for b in BASE_PER_RUN]
    YR = [residualize(y, b) for y, b in zip(Y_DEV, BASE_DEV)]


def loro_score(per_run_cols):
    Cr = [residualize(c.to(DEVICE), b) for c, b in zip(per_run_cols, BASE_DEV)]
    return loro_r2(YR, Cr)


def combine(*families):
    """Concatenate several per-run column families into one shared-beta set."""
    return [torch.cat(parts, 1) for parts in zip(*families)]


MODELS: dict[str, list[torch.Tensor]] = {}
if HAVE_DATA:
    MODELS = {
        "mot6": FAMILIES["mot6"],
        "mot12": FAMILIES["mot12"],
        "mot24": FAMILIES["mot24"],
        "motsim": FAMILIES["motsim"],
        "mot12+spin": combine(FAMILIES["mot12"], FAMILIES["spin"]),
        "mot12+bands": [band_cols(c) for c in FAMILIES["mot12"]],
        "motsim+bands": [band_cols(c) for c in FAMILIES["motsim"]],
        "motsim+spin": combine(FAMILIES["motsim"], FAMILIES["spin"]),
        "mot12+motsim": combine(FAMILIES["mot12"], FAMILIES["motsim"]),
    }
    for d in CFG["tv_degrees"][1:]:
        MODELS[f"mot12+tv{d}"] = [expand_time_varying(c, d) for c in FAMILIES["mot12"]]
        MODELS[f"motsim+tv{d}"] = [expand_time_varying(c, d) for c in FAMILIES["motsim"]]

    LORO = {}
    print(f"{'model':16s} {'cols':>5} {'held-out R2':>12}   per-fold")
    for name, cols in MODELS.items():
        r2, per_fold = loro_score(cols)
        LORO[name] = (r2.cpu(), per_fold.cpu())
        folds = "  ".join(f"{v:+.4f}" for v in per_fold.cpu().tolist())
        print(f"{name:16s} {cols[0].shape[1]:>5} {float(r2.mean()):>+12.5f}   {folds}")

In [ ]:
if HAVE_DATA and LORO:
    names = list(LORO)
    vals = [float(LORO[n][0].mean()) for n in names]
    order = np.argsort(vals)[::-1]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))

    ax[0].barh(
        [names[i] for i in order], [vals[i] for i in order],
        color=["tab:green" if vals[i] > 0 else "tab:red" for i in order],
    )
    ax[0].axvline(0, color="k", lw=1)
    ax[0].axvline(vals[names.index("mot12")], ls="--", lw=1, color="0.3", label="mot12")
    ax[0].set_xlabel("mean held-out $R^2$"); ax[0].legend(fontsize=8)
    ax[0].set_title("Stage 7 — the referee")

    # Per-fold spread: if one run dominates, the ranking is about that run.
    for i in order[:6]:
        ax[1].plot(LORO[names[i]][1].numpy(), "o-", lw=1, ms=4, label=names[i])
    ax[1].set_xlabel("held-out run"); ax[1].set_ylabel("$R^2$")
    ax[1].set_xticks(range(len(runs)))
    ax[1].axhline(0, color="k", lw=1)
    ax[1].legend(fontsize=7); ax[1].set_title("per-fold — check one run is not carrying it")
    fig.tight_layout(); plt.show()

### What to look for

**This is the number that decides it.** If `mot12+tv1` does not beat `mot12` here, the
time-varying gain is not worth the columns on this data, whatever Stage 4's F said. The same
for bands and for spin history.

**Check the per-fold panel.** Three runs means three folds; if one fold is wildly different
from the others, the ranking is an accident of which run got held out, and you should not
believe the ordering. More runs is the only fix.

**Negative $R^2$ everywhere for everything** means the motion→signal coupling simply does not
transfer across runs in this subject. That is an important finding on its own: it says the
conventional per-run refit is not a convenience, it is a necessity, and "one β per column"
was never the real problem — *"one β that means anything beyond this run"* was.

**`mot12+motsim` is there as a sanity check**, not a recommendation. If stacking them beats
both, the two are carrying different information; if it matches the better one, MotSim
subsumes the parameters as expected.

---

# Stage 8 — Does it help the thing we actually care about?

Held-out $R^2$ on the nuisance model is a proxy. The quantity anybody actually cares about is
whether the **task betas** get more trustworthy. So: fit each run separately, take the task
betas, and correlate them across runs. More reliable task betas under a nuisance model is the
outcome argument for that model.

This can move in the opposite direction to Stage 7, and when it does, *this* one wins. A
nuisance model that eats variance shared with the task will improve its own held-out $R^2$
while making the task estimate worse.

In [ ]:
def task_beta_reliability(per_run_cols) -> tuple[float, np.ndarray]:
    """Mean across-run correlation of the task beta maps. One number per model."""
    n_task = TASK.shape[1]
    betas = []
    for i in range(len(runs)):
        X = torch.cat(
            [TASK_PER_RUN[i], POLY[i], per_run_cols[i].cpu()], 1
        ).double()
        b = torch.linalg.pinv(X).to(Y_DEV[i].dtype).to(DEVICE) @ Y_DEV[i]
        betas.append(b[:n_task].cpu())
    per_cond = []
    for c in range(n_task):
        rs = []
        for i in range(len(runs)):
            for j in range(i + 1, len(runs)):
                a, b = betas[i][c].numpy(), betas[j][c].numpy()
                if a.std() > 0 and b.std() > 0:
                    rs.append(float(np.corrcoef(a, b)[0, 1]))
        per_cond.append(np.mean(rs) if rs else np.nan)
    return float(np.nanmean(per_cond)), np.array(per_cond)


RELI = {}
if HAVE_DATA:
    print(f"{'model':16s} {'task-beta reliability (mean across-run r)':>44s}")
    for name, cols in MODELS.items():
        m, per_cond = task_beta_reliability(cols)
        RELI[name] = (m, per_cond)
        print(f"{name:16s} {m:>44.4f}")
    base_r = RELI["mot12"][0]
    best = max(RELI, key=lambda k: RELI[k][0])
    print(
        f"\\nbest: {best} ({RELI[best][0]:.4f}), "
        f"{RELI[best][0] - base_r:+.4f} vs mot12"
    )

### What to look for

**Sign and size of the change relative to `mot12`.** A gain of +0.01 in across-run
correlation is noise; +0.05 is worth a second look; +0.1 on a well-powered task is a real
improvement in the estimate you will publish.

**Disagreement with Stage 7 is informative, not a problem.** A model that wins Stage 7 and
loses here is removing task-correlated variance — which is the standard hazard of any
data-driven nuisance set, and exactly what Patriat's "unlikely to contain signals of
interest (unless the task itself is correlated with the motion)" caveat is about. Look at
your task: if it involves speech, swallowing, button presses timed to stimuli, or anything
that makes people move *when the task says so*, expect this.

---

# Stage 9 — Shrinkage instead of selection

The whole difficulty is the degrees of freedom. But "how many columns can I afford?" is the
wrong question, because it forces a discrete choice on something continuous.

The better framing: put the base columns in an **unpenalized** block and the expansion in a
**penalized** one, then ask the data how hard to shrink. A useless expansion shrinks toward
zero instead of costing a clean degree of freedom; a useful one survives. [[Fractional
ridge]] parametrises the penalty by the fraction of the OLS coefficient norm retained — one
SVD covers the whole grid — and [[Frisch-Waugh-Lovell]] says the unpenalized block does not
have to enter the shrunk fit at all: residualise against it first, then ridge the remainder.

We score the whole frac grid with the same leave-one-run-out referee.

In [ ]:
FRACS = np.linspace(0.05, 1.0, 20)
RIDGE = {}

if HAVE_DATA:
    # Ridge fits nfracs x nvoxels coefficient sets; keep this section lighter.
    sub = slice(None) if Y[0].shape[1] <= 5000 else slice(0, 5000)

    for name in [f"mot12+tv{d}" for d in CFG["tv_degrees"][1:]] + ["mot12+bands"]:
        exp_cols = MODELS[name]
        # FWL: the unpenalized block is baseline + the plain mot12 columns.
        unpen = [torch.cat([b, c.to(DEVICE)], 1) for b, c in zip(BASE_DEV, FAMILIES["mot12"])]
        Yr = [residualize(y[:, sub], u) for y, u in zip(Y_DEV, unpen)]
        Cr = [residualize(c.to(DEVICE), u) for c, u in zip(exp_cols, unpen)]

        curve = np.zeros(len(FRACS))
        for k in range(len(runs)):
            Xtr = torch.cat([Cr[i] for i in range(len(runs)) if i != k], 0)
            Ytr = torch.cat([Yr[i] for i in range(len(runs)) if i != k], 0)
            coefs = _fit_ridge_multiple_fracs(Xtr, Ytr, FRACS, DEVICE)  # (k, nfrac, V)
            ss_tot = (Yr[k] ** 2).sum(0).clamp_min(1e-30)
            for fi in range(len(FRACS)):
                resid = Yr[k] - Cr[k] @ coefs[:, fi, :]
                curve[fi] += float((1.0 - (resid**2).sum(0) / ss_tot).mean()) / len(runs)
        RIDGE[name] = curve
        best = int(np.argmax(curve))
        print(
            f"{name:14s} best frac={FRACS[best]:.2f} -> held-out R2 {curve[best]:+.5f} "
            f"(OLS, frac=1: {curve[-1]:+.5f})"
        )

    if RIDGE:
        plt.figure(figsize=(6, 3.6))
        for name, curve in RIDGE.items():
            plt.plot(FRACS, curve, "o-", ms=3, lw=1.2, label=name)
        plt.axhline(0, color="k", lw=1)
        plt.xlabel("fraction of OLS norm retained"); plt.ylabel("held-out $R^2$ of the expansion")
        plt.title("Stage 9 — shrinkage beats selection")
        plt.legend(fontsize=8); plt.tight_layout(); plt.show()

### What to look for

**A peak at frac < 1 means shrinkage is buying you something** — the OLS fit of the
expansion was overfitting, and a penalised version of the same columns generalises better.
That is the constructive answer to "I cannot afford these degrees of freedom": you can, if
you do not insist on fitting them freely.

**A curve that is flat and at or below zero across the whole grid** is the cleanest possible
negative result. There is no shrinkage at which these columns help. Stop.

**A curve that rises monotonically to frac = 1** says the expansion is genuinely
low-variance and does not need the penalty — unusual, and worth double-checking against
Stage 7.

---

# Stage 10 — Does MotSim already do this?

MotSim is approximately a second-order expansion in pose, weighted by anatomy. So it should
already absorb a chunk of whatever the raw parameters were missing on the *nonlinearity*
axis. The question here is whether it also absorbs the **time** and **frequency** axes, which
it has no mechanism for.

The test: compare the held-out gain from expanding `mot12` against the gain from expanding
`motsim`. If the gains are similar, the expansions are capturing something orthogonal to what
MotSim does, and are worth having on top of it. If expanding MotSim gains nothing, MotSim's
PCs were already spanning it.

In [ ]:
if HAVE_DATA and LORO:
    print(f"{'expansion':10s} {'on mot12':>18s} {'on motsim':>18s}")
    rows = [("tv%d" % d, f"mot12+tv{d}", f"motsim+tv{d}") for d in CFG["tv_degrees"][1:]]
    rows += [("bands", "mot12+bands", "motsim+bands"), ("spin", "mot12+spin", "motsim+spin")]
    base12, basems = float(LORO["mot12"][0].mean()), float(LORO["motsim"][0].mean())
    for tag, a, b in rows:
        if a not in LORO or b not in LORO:
            continue
        ga = float(LORO[a][0].mean()) - base12
        gb = float(LORO[b][0].mean()) - basems
        print(f"{tag:10s} {ga:>+18.5f} {gb:>+18.5f}")
    print(
        f"\\nbaselines: mot12 {base12:+.5f}, motsim {basems:+.5f}\\n"
        "Similar gains in both columns => the expansion is orthogonal to what MotSim does.\\n"
        "A gain on mot12 that vanishes on motsim => MotSim already spanned it."
    )

---

# Summary

In [ ]:
if HAVE_DATA and LORO:
    base12 = float(LORO["mot12"][0].mean())
    base_r = RELI["mot12"][0]
    print(f"{'model':16s} {'cols':>5} {'heldout R2':>12} {'vs mot12':>10} "
          f"{'task rel.':>10} {'vs mot12':>10}")
    print("-" * 68)
    for name in MODELS:
        r2 = float(LORO[name][0].mean())
        rel = RELI[name][0]
        print(
            f"{name:16s} {MODELS[name][0].shape[1]:>5} {r2:>+12.5f} {r2 - base12:>+10.5f} "
            f"{rel:>10.4f} {rel - base_r:>+10.4f}"
        )
    print("-" * 68)
    print("'vs mot12' is the only column worth acting on. Positive in BOTH is a win.")

## What we learned

Fill this in from your own numbers — but the shape of the conclusion is one of four, and it
is worth naming which one you got:

1. **Nothing clears the surrogates and nothing improves held-out $R^2$.** The stationary,
   single-β model is adequate on this data. That is a real answer, and it means the effort
   belongs somewhere else (better moco, MEDIC, censoring).
2. **In-sample F is large but held-out $R^2$ does not improve.** The expansions are fitting
   run-specific noise. This is the outcome the whole notebook is designed to catch, and it is
   the most likely one for a well-behaved subject.
3. **Held-out $R^2$ improves but task-beta reliability drops.** The expansion is eating
   task-correlated variance. Do not ship it. Look at whether your task makes people move.
4. **Both improve.** You have found something. Now check it replicates in a second subject
   before believing it, because everything here is n=1.

## What we cannot learn from this

Being explicit about the ceiling, because several of these are easy to forget once a map
lights up:

- **Whether a drifting β means the coupling drifted or the motion estimate did.** Moco is
  anchored to one base volume; a run that drifts away from it is registered across a larger
  displacement by the end. Stage 10 gives a partial handle, nothing more. The real answer
  needs a second moco with a different base, or an external tracker.
- **Whether the respiratory band is respiration.** Without a belt or a pulse-ox trace, the
  frequency is the only evidence, and "there is a peak where respiration usually is" is
  suggestive, not proof. If you have physio recordings, correlate the band against them —
  that turns this from an inference into a measurement.
- **Anything about intra-volume motion.** Everything here is between-volume by construction.
  MotSim cannot reach it, the expansions cannot reach it, and censoring remains the only tool
  that does — which is exactly why Patriat found ICA-AROMA beat MotSim *without* censoring
  and matched it *with*.
- **Whether this generalises.** One subject, one task, one acquisition. [[Data Variety]] is
  not a slogan: TR, tSNR, voxel size, number of runs and how much the subject moved will each
  move these numbers around. The TR gate in Stage 3 is the clearest example — the same
  analysis on TR=2.6s data cannot even ask the frequency question.
- **Whether a better *model* beats a better *measurement*.** Everything here rearranges six
  numbers estimated from low-resolution EPI. A prospective tracker, a field camera, or a
  multi-echo acquisition changes the input rather than the regression, and would likely
  dominate any of it.

## If you want to take it further

- **More runs.** Three folds is the minimum and the per-fold panel usually shows it. Six runs
  makes the LORO ranking believable.
- **A second subject**, run end-to-end unchanged, before believing any ordering.
- **Per-voxel expansion order.** The maps in Stages 4 and 5 almost certainly are not uniform.
  A [[Per-voxel optimization]] — expansion degree chosen per voxel by held-out $R^2$, with
  the order map saved as a diagnostic — is the natural next step, and it is how this codebase
  handles every other voxel-wise hyperparameter.
- **Physio.** A respiratory belt turns Stage 5 from an inference into a measurement.